# 金融數據學 Python：程式邏輯與選股思維

> 任務十：剔除成交量過低的公司 | The One Who Is Popular In The Market

[郭耀仁](https://hahow.in/@tonykuoyj?tr=tonykuoyj) | <yaojenkuo@datainpoint.com>

## 執行下方儲存格將所需資料下載到工作目錄

In [1]:
!wget -Nq https://raw.githubusercontent.com/datainpoint/classroom-hahow-learn-python-with-stock-data/refs/heads/main/datasets/trading_volumes.xlsx
!wget -Nq https://raw.githubusercontent.com/datainpoint/classroom-hahow-learn-python-with-stock-data/refs/heads/main/datasets/trading_volumes.db

## 1. 載入 `trading_volumes.db` 中的 `tickers`、`volumes` 兩張資料表成為 `DataFrame`，並分別命名為 `tickers`、`volumes`，參考下列 SQL 敘述的範例：

```python
table_name = "tickers"
sql_query = f"""
SELECT *
  FROM {table_name};
"""
table_name = "volumes"
sql_query = f"""
SELECT *
  FROM {table_name};
"""
```

In [2]:
import sqlite3
import pandas as pd
file_path = "trading_volumes.db"
con = sqlite3.connect(file_path)
sql_query ="""
SELECT *
    FROM tickers;
    """
tickers = pd.read_sql(sql_query , con)
sql_query ="""
SELECT *
    FROM volumes;
    """
volumes = pd.read_sql(sql_query , con)
con.close()
print(tickers.head())
print(volumes.head())

   id     ticker
0   1    0057.TW
1   2  00675L.TW
2   3   1264.TWO
3   4    1303.TW
4   5    1476.TW
   ticker_id   traded_on  volume
0         10  2026-09-29  402183
1         10  2026-09-30  340140
2         10  2026-10-01  293480
3         10  2026-10-02  472350
4         10  2026-10-05  456246


## 2. 透過 `trading_volumes.xlsx` 計算資料期間交易均量的 25 分位數，並將計算結果命名為 `first_quantile_xlsx`

In [3]:
file_path = "trading_volumes.xlsx"
excel_file = pd.ExcelFile(file_path) #讀取檔案
stock_tickers = excel_file.sheet_names #找出每個活頁簿名稱
print(stock_tickers)
print(len(stock_tickers))
average_trading_volumes = list()
for stock_ticker in stock_tickers:
    df = pd.read_excel(excel_file , sheet_name = stock_ticker)
    volume_mean = df["volume"].mean()
    average_trading_volumes.append(volume_mean)
ser = pd.Series(average_trading_volumes)
first_quantile_xlsx = ser.quantile(0.25)
print(first_quantile_xlsx)

['2059.TW', '3443.TW', '3653.TW', '3008.TW', '7769.TW', '2383.TW', '6515.TW', '2454.TW', '3661.TW', '3017.TW', '6805.TW', '2330.TW', '2360.TW', '7750.TW', '3533.TW', '6669.TW', '2345.TW', '3665.TW', '8996.TW', '2308.TW', '5269.TW', '6442.TW', '7610.TW', '3406.TW', '8046.TW', '6446.TW', '2368.TW', '1590.TW', '2404.TW', '3037.TW', '3189.TW', '3167.TW', '2357.TW', '6409.TW', '6781.TW', '1560.TW', '6531.TW', '6526.TW', '2379.TW', '7822.TW', '8210.TW', '6944.TW', '6213.TW', '3026.TW', '3711.TW', '2395.TW', '3563.TW', '6139.TW', '6691.TW', '3583.TW', '3034.TW', '1519.TW', '6196.TW', '7711.TW', '6672.TW', '5434.TW', '4958.TW', '2327.TW', '3044.TW', '2207.TW', '6415.TW', '2455.TW', '6830.TW', '6715.TW', '2467.TW', '2408.TW', '6957.TW', '6789.TW', '3450.TW', '8021.TW', '7799.TW', '6491.TW', '7795.TW', '3532.TW', '6933.TW', '4583.TW', '6831.TW', '2492.TW', '6472.TW', '6414.TW', '8464.TW', '6451.TW', '2376.TW', '4766.TW', '2049.TW', '2382.TW', '6197.TW', '00675L.TW', '6426.TW', '7749.TW', '0057.T

## 3. 透過 `volumes` 計算資料期間交易均量的 25 分位數，並將計算結果命名為 `first_quantile_db`，比較一下與前題的計算結果是否相同。

In [4]:
average_trading_volume_by_ticker_id = volumes.groupby("ticker_id")["volume"].mean().reset_index()
first_quantile_db = average_trading_volume_by_ticker_id["volume"].quantile(0.25)
print(first_quantile_db)
print(average_trading_volume_by_ticker_id)

426194.5125
     ticker_id       volume
0            1     26996.10
1            2   1099166.15
2            3     13421.25
3            4  94932375.25
4            5   1606046.15
..         ...          ...
195        196   2861167.00
196        197   2273479.25
197        198  12487883.45
198        199    477179.05
199        200   3716124.00

[200 rows x 2 columns]


## 4. 判斷 `tickers` 中哪些上市上櫃公司資料期間交易均量大於 25 分位數，並將判斷結果轉換為外型 `(200, 2)` 的資料框命名為 `is_traded_in_high_volume_df`

```
        ticker  is_traded_in_high_volume
0    00631L.TW                      True
1     1264.TWO                     False
2      1476.TW                      True
3      1477.TW                      True
4      1519.TW                      True
..         ...                       ...
195   8416.TWO                     False
196    8464.TW                      True
197    8499.TW                      True
198   8932.TWO                      True
199    8996.TW                      True

[200 rows x 2 columns]
```

In [5]:
merge = pd.merge(average_trading_volume_by_ticker_id, tickers , left_on="ticker_id",right_on="id")
bool_index = merge["volume"] > first_quantile_db
is_traded_in_high_volume_df = pd.DataFrame({"ticker":merge["ticker"],"is_traded_in_high_volume":bool_index})
print(is_traded_in_high_volume_df)

        ticker  is_traded_in_high_volume
0      0057.TW                     False
1    00675L.TW                      True
2     1264.TWO                     False
3      1303.TW                      True
4      1476.TW                      True
..         ...                       ...
195   8227.TWO                      True
196   8299.TWO                      True
197   8358.TWO                      True
198    8464.TW                      True
199    8996.TW                      True

[200 rows x 2 columns]


## 5. 使用布林索引，將 `is_traded_in_high_volume_df` 中上市上櫃公司資料期間交易均量大於 25 分位數的觀測值挑選出來，並將挑選後的資料框命名為 `traded_in_high_volume_df`

In [6]:
traded_in_high_volume_df = is_traded_in_high_volume_df[bool_index]
print(traded_in_high_volume_df)

        ticker  is_traded_in_high_volume
1    00675L.TW                      True
3      1303.TW                      True
4      1476.TW                      True
5      1519.TW                      True
6      1560.TW                      True
..         ...                       ...
195   8227.TWO                      True
196   8299.TWO                      True
197   8358.TWO                      True
198    8464.TW                      True
199    8996.TW                      True

[150 rows x 2 columns]


## 重新啟動並執行所有儲存格進行批改測試，暸解是否完成所有任務項目

Runtime -> Restart session and run all -> Yes -> Run anyway.

In [7]:
import unittest

class TestMissionTen(unittest.TestCase):
    def test_01(self):
        self.assertEqual(tickers.shape[0], 200)
        self.assertEqual(tickers.shape[1], 2)
        self.assertEqual(volumes.shape[1], 3)
    def test_02(self):
        self.assertTrue(isinstance(first_quantile_xlsx, float))
        self.assertEqual(first_quantile_xlsx, first_quantile_db)
    def test_03(self):
        self.assertTrue(isinstance(first_quantile_db, float))
        self.assertEqual(first_quantile_xlsx, first_quantile_db)
    def test_04(self):
        self.assertEqual(is_traded_in_high_volume_df.shape[0], 200)
        self.assertEqual(is_traded_in_high_volume_df.shape[1], 2)
    def test_05(self):
        self.assertLessEqual(traded_in_high_volume_df.shape[0], 150)
        self.assertEqual(traded_in_high_volume_df.shape[1], 2)

suite = unittest.TestLoader().loadTestsFromTestCase(TestMissionTen)
runner = unittest.TextTestRunner(verbosity=2)
test_results = runner.run(suite)
number_of_failures = len(test_results.failures)
number_of_errors = len(test_results.errors)
number_of_test_runs = test_results.testsRun
number_of_successes = number_of_test_runs - (number_of_failures + number_of_errors)
chapter_name = "任務十：剔除成交量過低的公司 | The One Who Is Popular In The Market"
print(f"恭喜你！您在「{chapter_name}」中的 {number_of_test_runs} 個任務項目完成了 {number_of_successes} 個！")

test_01 (__main__.TestMissionTen.test_01) ... ok
test_02 (__main__.TestMissionTen.test_02) ... ok
test_03 (__main__.TestMissionTen.test_03) ... ok
test_04 (__main__.TestMissionTen.test_04) ... ok
test_05 (__main__.TestMissionTen.test_05) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.034s

OK


恭喜你！您在「任務十：剔除成交量過低的公司 | The One Who Is Popular In The Market」中的 5 個任務項目完成了 5 個！
